# 04 · Chunking Strategies
### Practical LLM Engineering — Module 03: Embeddings & Retrieval

> **Objectives**
> - Why chunking decisions are the most impactful RAG engineering choice
> - Fixed-size, sentence-boundary, paragraph, recursive, and semantic chunking
> - Chunk overlap and its effect on recall
> - Late chunking and the context-window tradeoff
> - Measuring chunking quality with recall@k experiments

---


## 1. Overview

Before any document can be searched, it must be **chunked** — split into pieces small enough for an embedding model and an LLM context window.

Chunking is the highest-leverage decision in a RAG pipeline:
- Too large → poor embedding quality, fits fewer docs in LLM context
- Too small → loses context, requires more chunks to answer a question
- Bad boundaries → cuts sentences mid-thought, hurts coherence

### Chunking strategies (from simple to sophisticated)

```
1. Fixed-size      — split every N characters
2. Sentence        — split at sentence boundaries
3. Paragraph       — split at paragraph breaks
4. Recursive       — try paragraphs → sentences → words → chars
5. Semantic        — merge until embedding similarity drops
6. Late chunking   — embed full document, chunk embeddings post-hoc
```


## 2. Setup

In [ ]:
# !pip install sentence-transformers nltk numpy matplotlib -q
# import nltk; nltk.download('punkt_tab')
import re, math, textwrap
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize

print("✅ Libraries ready")

class MockEmbedder:
    def __init__(self, dim=64, seed=42):
        self.dim   = dim
        self._proj = np.random.RandomState(seed).randn(10_000, dim).astype(np.float32)
    def _hash(self, w): return int(abs(hash(w.lower())) % 10_000)
    def embed_one(self, text):
        tokens = text.lower().split()
        if not tokens: return np.zeros(self.dim, dtype=np.float32)
        v = np.mean([self._proj[self._hash(t)] for t in tokens], axis=0)
        return v / max(np.linalg.norm(v), 1e-9)
    def embed(self, texts):
        return normalize(np.array([self.embed_one(t) for t in texts]))

embedder = MockEmbedder(dim=64)


In [ ]:
LONG_DOCUMENT = """
Retrieval-Augmented Generation (RAG) is a technique that enhances large language models
by grounding their responses in retrieved external knowledge. Instead of relying solely
on parametric knowledge baked into model weights during training, RAG systems dynamically
fetch relevant documents at inference time.

The RAG pipeline consists of two main phases: offline indexing and online retrieval.
During offline indexing, documents are chunked, embedded, and stored in a vector database.
This process happens once and is amortised over many queries.

During online retrieval, a user query is embedded and used to search the vector store.
The top-k most similar chunks are retrieved and inserted into the LLM prompt as context.
The LLM then generates an answer grounded in the retrieved passages.

Chunking is perhaps the most critical step in the indexing pipeline. The chunk size
determines how much context is embedded into a single vector. Too small, and the
embedding lacks sufficient semantic signal. Too large, and the embedding averages over
multiple topics, reducing precision.

Common chunking sizes range from 128 to 1024 tokens depending on the use case.
Question-answering tasks often benefit from smaller chunks (128-256 tokens) that are
highly targeted. Summarisation tasks may use larger chunks (512-1024 tokens) to preserve
more narrative context in each embedding.

Overlap between adjacent chunks — typically 10-20% of the chunk size — helps ensure
that information near chunk boundaries is represented in at least one chunk's embedding.
Without overlap, a key sentence split across a boundary might be missed by both chunks.

Sentence-boundary chunking ensures that chunks do not cut mid-sentence, which would
produce incoherent embeddings. This is achieved by splitting at sentence terminators
detected by an NLP tokeniser and then merging or splitting to meet the token budget.

Semantic chunking takes a more adaptive approach: it monitors the cosine similarity
between adjacent sentence embeddings and introduces a new chunk boundary wherever
similarity drops below a threshold. This naturally groups topically coherent content
into the same chunk without hard token limits.
"""
print(f'Document: {len(LONG_DOCUMENT)} characters, {len(LONG_DOCUMENT.split())} words')

## 3. Chunking Strategies

In [ ]:
# ── Strategy 1: Fixed-size character chunking ─────────────────────────
def chunk_fixed(text: str, chunk_size: int = 300,
                overlap: int = 50) -> list[str]:
    """Split text into fixed-size character chunks with overlap."""
    chunks, start = [], 0
    while start < len(text):
        end   = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


# ── Strategy 2: Sentence-boundary chunking ────────────────────────────
def split_sentences(text: str) -> list[str]:
    """Split text into sentences using regex (no NLTK dependency)."""
    # Split on .!? followed by whitespace and capital letter
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text.strip())
    return [s.strip() for s in sentences if s.strip()]

def chunk_sentences(text: str, max_tokens: int = 80,
                     overlap_sents: int = 1) -> list[str]:
    """Merge sentences into chunks respecting a token budget."""
    sentences = split_sentences(text)
    chunks    = []
    buf, buf_tokens = [], 0

    for sent in sentences:
        n = len(sent.split())
        if buf and buf_tokens + n > max_tokens:
            chunks.append(" ".join(buf))
            buf       = buf[-overlap_sents:]   # keep overlap
            buf_tokens = sum(len(s.split()) for s in buf)
        buf.append(sent)
        buf_tokens += n

    if buf:
        chunks.append(" ".join(buf))
    return chunks


# ── Strategy 3: Paragraph chunking ───────────────────────────────────
def chunk_paragraphs(text: str, max_tokens: int = 150) -> list[str]:
    """Split on paragraph breaks, merge short ones to meet token budget."""
    paras  = [p.strip() for p in re.split(r'\n{2,}', text) if p.strip()]
    chunks = []
    buf, buf_tokens = [], 0

    for para in paras:
        n = len(para.split())
        if buf and buf_tokens + n > max_tokens:
            chunks.append(" ".join(buf))
            buf, buf_tokens = [], 0
        buf.append(para)
        buf_tokens += n

    if buf:
        chunks.append(" ".join(buf))
    return chunks


# ── Strategy 4: Recursive chunking ────────────────────────────────────
def chunk_recursive(text: str, chunk_size: int = 300,
                     separators: list[str] = None) -> list[str]:
    """
    Try splitting by separators in order; recurse on pieces that are
    still too large. Mirrors LangChain RecursiveCharacterTextSplitter.
    """
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    if len(text) <= chunk_size:
        return [text.strip()] if text.strip() else []

    for sep in separators:
        parts = text.split(sep) if sep else list(text)
        chunks = []
        current = ""
        for part in parts:
            candidate = current + (sep if current else "") + part
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    chunks.append(current.strip())
                current = part
        if current:
            chunks.append(current.strip())

        # Recurse on any still-too-large chunks
        result = []
        for c in chunks:
            if len(c) > chunk_size:
                result.extend(chunk_recursive(c, chunk_size, separators[1:]))
            elif c:
                result.append(c)
        if result:
            return result

    return [text[:chunk_size].strip()]


# ── Strategy 5: Semantic chunking ─────────────────────────────────────
def chunk_semantic(text: str, embedder,
                    threshold: float = 0.8,
                    min_chunk_tokens: int = 30) -> list[str]:
    """
    Group sentences into chunks based on embedding similarity.
    A new chunk starts when similarity to the current chunk drops below threshold.
    """
    sentences = split_sentences(text)
    if not sentences:
        return []

    sent_vecs   = embedder.embed(sentences)   # (N, d)
    chunks      = [sentences[0]]
    chunk_vecs  = [sent_vecs[0]]

    for i in range(1, len(sentences)):
        # Current chunk centroid
        centroid = np.mean(chunk_vecs, axis=0)
        centroid /= max(np.linalg.norm(centroid), 1e-9)
        sim = float(np.dot(centroid, sent_vecs[i]))

        if sim >= threshold or len(chunks[-1].split()) < min_chunk_tokens:
            chunks[-1] += " " + sentences[i]
            chunk_vecs.append(sent_vecs[i])
        else:
            chunks.append(sentences[i])
            chunk_vecs = [sent_vecs[i]]

    return [c.strip() for c in chunks if c.strip()]


# ── Compare all strategies ─────────────────────────────────────────────
strategies = {
    "Fixed-size (300ch)":   chunk_fixed(LONG_DOCUMENT, 300, 50),
    "Sentence (80tok)":     chunk_sentences(LONG_DOCUMENT, 80, 1),
    "Paragraph (150tok)":   chunk_paragraphs(LONG_DOCUMENT, 150),
    "Recursive (300ch)":    chunk_recursive(LONG_DOCUMENT, 300),
    "Semantic (sim≥0.75)":  chunk_semantic(LONG_DOCUMENT, embedder, 0.75),
}

print(f"{'Strategy':<25} {'n_chunks':>9} {'avg_tokens':>11} {'min':>6} {'max':>6}")
print("-" * 62)
for name, chunks in strategies.items():
    token_counts = [len(c.split()) for c in chunks]
    print(f"{name:<25} {len(chunks):>9} {np.mean(token_counts):>11.1f} "
          f"{min(token_counts):>6} {max(token_counts):>6}")


In [ ]:
# ── Chunk size distribution visualisation ─────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle("Chunk token-count distributions by strategy", fontsize=11)
axes_flat = axes.flatten()

for ax, (name, chunks) in zip(axes_flat, strategies.items()):
    token_counts = [len(c.split()) for c in chunks]
    ax.hist(token_counts, bins=max(3, len(token_counts)//2+1),
            color="#3498db", edgecolor="white", alpha=0.85)
    ax.axvline(np.mean(token_counts), color="#e74c3c", linestyle="--",
               lw=1.5, label=f"mean={np.mean(token_counts):.0f}")
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("Tokens per chunk"); ax.set_ylabel("Count")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

axes_flat[-1].set_visible(False)
plt.tight_layout()
plt.show()


## 4. The Overlap Effect

Overlap ensures that information near chunk boundaries is captured. Without overlap, a sentence split across two chunks contributes to neither chunk's embedding:

```
Chunk A:  "...The quick brown fox jumps over the [lazy dog.] "  ← "lazy dog" appears
Chunk B:  "A new topic begins here..."

Without overlap: "jumps over the lazy dog" has low embedding salience in A and none in B
With overlap:    B starts mid-A, so "lazy dog" appears twice → higher recall
```


In [ ]:
# ── Overlap sweep: recall vs overhead ────────────────────────────────
def build_chunk_index(chunks: list[str], embedder) -> tuple[np.ndarray, list[str]]:
    vecs = embedder.embed(chunks)
    return vecs, chunks

def retrieval_recall(query: str, vecs: np.ndarray, chunks: list[str],
                      embedder, relevant_substrings: list[str], k: int = 3) -> float:
    q_vec   = embedder.embed([query])[0]
    scores  = vecs @ q_vec
    top_k   = np.argsort(-scores)[:k]
    found   = " ".join(chunks[i] for i in top_k).lower()
    return sum(1 for s in relevant_substrings if s.lower() in found) / len(relevant_substrings)

# Test query and expected substrings that should be in top-3 chunks
TEST_CASES = [
    ("What is chunk overlap and why does it matter?",
     ["overlap", "boundary", "chunk"]),
    ("How does semantic chunking work?",
     ["semantic", "similarity", "threshold"]),
    ("What chunk size works best for QA?",
     ["128", "256", "question"]),
]

overlaps = [0, 20, 40, 80, 120]
recalls_by_overlap = []

for overlap in overlaps:
    chunks = chunk_fixed(LONG_DOCUMENT, 300, overlap)
    vecs, _ = build_chunk_index(chunks, embedder)
    avg_recall = np.mean([
        retrieval_recall(q, vecs, chunks, embedder, subs, k=3)
        for q, subs in TEST_CASES
    ])
    overhead = (overlap * len(chunks)) / max(len(LONG_DOCUMENT), 1)
    recalls_by_overlap.append((overlap, avg_recall, overhead, len(chunks)))

print("Overlap sweep (fixed-size 300ch chunks):")
print(f"{'Overlap':>10} {'n_chunks':>9} {'Recall@3':>10} {'Storage overhead':>18}")
print("-" * 52)
for ov, rec, oh, nc in recalls_by_overlap:
    print(f"{ov:>10} {nc:>9} {rec:>10.3f} {oh:>17.1%}")


## 5. Chunking Evaluation

In [ ]:
# ── Evaluate each strategy on a small retrieval task ─────────────────
EVAL_QUERIES = [
    ("What is RAG and what problem does it solve?",
     ["retrieval-augmented", "external knowledge", "grounding"]),
    ("Describe the online retrieval phase of RAG.",
     ["query is embedded", "vector store", "top-k"]),
    ("What is the recommended chunk size for QA tasks?",
     ["128", "256", "question-answering"]),
    ("What is semantic chunking and how does it differ?",
     ["cosine similarity", "adaptive", "threshold"]),
]

print("Strategy evaluation — Recall@3 on 4 queries:")
print(f"{'Strategy':<25} {'Avg Recall@3':>13} {'n_chunks':>9} {'Avg tokens':>11}")
print("-" * 62)

for name, chunks in strategies.items():
    vecs, _ = build_chunk_index(chunks, embedder)
    recall   = np.mean([
        retrieval_recall(q, vecs, chunks, embedder, subs, k=3)
        for q, subs in EVAL_QUERIES
    ])
    tc = [len(c.split()) for c in chunks]
    print(f"{name:<25} {recall:>13.3f} {len(chunks):>9} {np.mean(tc):>11.1f}")


## 6. Engineering Insights & Decision Guide

### Chunk size selection

| Use case | Recommended chunk size | Rationale |
|---|---|---|
| Factual QA | 128–256 tokens | Precise retrieval, single-fact answers |
| Summarisation | 512–1024 tokens | Need narrative context |
| Code search | Function-level | Natural semantic unit |
| Legal / long-form | 256–512 + metadata | Preserve argument structure |

### Strategy selection

```
Simple docs, fast iteration:      Fixed-size with overlap (200-300 chars, 10-20% overlap)
High-quality QA:                  Sentence-boundary with overlap
Structured docs (articles, reports): Paragraph chunking
Mixed doc types:                  Recursive (LangChain default)
Highest quality, GPU available:   Semantic chunking
```

### Late chunking (jina-embeddings-v2)

Instead of chunking → embedding, embed the full document using a long-context model,
then extract per-chunk vectors by averaging token embeddings within chunk spans.
This preserves full-document context in every chunk embedding — but requires a long-context
encoder and is more expensive.

